# 01 EDA：SECOM 原始資料探索

這份 notebook 記錄早期探索原始資料時的過程，包含基本描述統計，以及
Time 欄六次時間倒退的偵錯推理過程。**結論寫在 README，這裡留過程**。

正式的清理／切分邏輯在 `src/data.py`、`src/features.py`，跟這裡的探
索程式碼是分開的——這裡算出來的部分數字（常數欄、缺失率）是在**全部
1567 筆資料**上算的，純粹用來認識資料，不是正式建模用的統計量；正式
建模的對應數字必須只從訓練集算，見 CLAUDE.md「切分優先」原則。

## 資料基本形狀

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/uci-secom.csv")
df.shape

執行結果：`(1567, 592)`——1567 列、592 欄（`Time` + 590 個感測器欄
`0`~`589` + `Pass/Fail`），跟 CLAUDE.md 描述的資料規模一致。

## 標籤分布

In [ ]:
df["Pass/Fail"].value_counts()

執行結果：`-1`（Pass）1463 筆、`1`（Fail）104 筆，Fail 佔 6.64%——
嚴重的類別不平衡。這是後面 XGBoost 要用 `scale_pos_weight`、閾值不能
用預設 0.5、以及評估要看 PR-AUC 而非 accuracy 的根本原因。

## 缺失值

In [ ]:
total_missing = df.isna().sum().sum()
missing_rate = df.isna().mean()
high_missing = (missing_rate > 0.5).sum()

total_missing, int(high_missing)

執行結果：全表缺失值共 41951 個；缺失率超過 50% 的欄有 28 欄。這是在
**全部 1567 筆資料**上算的探索性數字——正式清理時「缺失率 > 50%」的
門檻只能用訓練集算（見 `src/features.py` 的
`find_high_missing_columns`），訓練集上的結果是 24 欄。兩者不同純粹
是因為算的資料範圍不一樣，不代表哪個算錯了。

## 常數欄

In [ ]:
sensor_cols = df.columns[1:-1]
nunique = df[sensor_cols].nunique(dropna=True)
(nunique <= 1).sum()

執行結果：全部資料上有 116 欄是常數欄（`nunique<=1`）。**這只是探索
性的全資料集數字**——正式清理時常數欄只能從訓練集判斷
（`src/features.py` 的 `find_constant_columns`），訓練集上實際是
122 欄。差異方向是合理的：全資料集判定為常數的欄，在任何子集裡也必
然是常數；但訓練子集裡看起來是常數的欄，可能只是因為它僅有的變化剛
好落在被切到測試集的那段時間裡，放回全資料集看就不是常數了。**兩個
數字都對，只是算的資料範圍不同，正式建模只採用訓練集版本。**

## Time 欄時間倒退的偵錯過程

原始檔按列序看，時間並非單調遞增。以下記錄當時排查這件事的完整過
程：先確認倒退確實存在，再逐一排除可能的原因。

### 第一步：確認倒退真的存在

In [ ]:
t = pd.to_datetime(df["Time"])
back_idx = t.index[t.diff() < pd.Timedelta(0)]

for i in back_idx:
    print(f"index {i}: {t[i-1]} -> {t[i]}  (倒退 {t[i-1] - t[i]})")

執行結果：整份資料有 6 個倒退點，最大一次倒退 264 天 23 小時 38 分
（index 1208）。列序不是嚴格按時間遞增的。

### 第二步：會不會是日期解析歧義造成的？

原始字串長這樣：`2008-07-19 11:55:00`。第一個懷疑是 `pd.to_datetime`
會不會把「月」跟「日」猜反——例如把 `07` 跟 `19` 誤判方向，這種格式
如果兩個數字都 ≤12 會有歧義，`dayfirst` 參數正是控制這個猜測方向的。
用兩行程式碼直接排除這個假設：

In [ ]:
t_dayfirst_true = pd.to_datetime(df["Time"], dayfirst=True)
t_dayfirst_false = pd.to_datetime(df["Time"], dayfirst=False)

t_dayfirst_true.equals(t_dayfirst_false)

執行結果：`True`——兩種解析方式得到完全相同的時間序列，倒退次數同樣
都是 6 次。原因是原始字串已經是無歧義的 ISO 8601 格式
（`YYYY-MM-DD HH:MM:SS`），年份放最前面，`dayfirst` 只影響 `DD/MM`
這種兩個數字都在前面、可能混淆月日順序的格式，對 ISO 格式完全不生
效。**排除解析歧義假設。**

## 結論

6 個倒退點確認不是解析問題，推測是資料本身的真實結構（可能是 6 段
各自依時間排序的區塊被直接串接而成的檔案）。最終決定維持「載入後用
stable sort 依 Time 統一重新排序，再切分訓練/測試」的做法，不特別處理
這 6 個區塊邊界——完整理由與決定記錄在 [README.md](../README.md)。